# Advanced Collections: `ChainMap`, `deque`, `defaultdict` and `Counter`, Measured with `timeit`

## Merging Configurations

An application reads its configuration at three levels: its default values (`defaults`), the user's file (`user`) and the command-line options (`command_line`). Each level overrides the previous ones.

1. With `collections.ChainMap`, build a view `configuration` of the three levels, then print `dict(configuration)`. Expected: `{'font_size': 11, 'left_margin': 0, 'right_margin': 2, 'top_margin': 3}`.
2. Also build, with the `|` operator, a dictionary `merged` with the same values. The user then sets their right margin to 4: modify `user`, then print `right_margin` in `configuration` and in `merged`. Expected: `4 2`.
3. With `new_child()`, add to `configuration` a temporary override of the font size to 14, in a new view `temporary`, then print the font size in `temporary` and in `configuration`. Expected: `14 11`.

**Useful documentation**

- [`ChainMap.new_child`](https://docs.python.org/3/library/collections.html#collections.ChainMap.new_child)

In [ ]:
defaults = dict(font_size=10, left_margin=1, right_margin=2)
user = dict(font_size=11, top_margin=3)
command_line = dict(left_margin=0)


# Your code here

### Additional hints

`ChainMap(first, second, ...)` looks a key up in its dictionaries in the order of the arguments and returns the first one found: the level that wins therefore comes first, here `command_line`. With `a | b`, it's the other way round: the value of `b`, the last operand, wins, and the result is a new dictionary. `view.new_child(d)` returns a new `ChainMap` where `d` comes before all the dictionaries of `view`, without changing `view`.

```python
import collections


configuration = collections.ChainMap(...)
print(dict(configuration))

merged = ...
user["right_margin"] = 4
print(configuration["right_margin"], merged["right_margin"])

temporary = configuration.new_child(...)
print(temporary["font_size"], configuration["font_size"])
```

### Solution

In [ ]:
import collections


configuration = collections.ChainMap(command_line, user, defaults)
print(dict(configuration))

merged = defaults | user | command_line
user["right_margin"] = 4
print(configuration["right_margin"], merged["right_margin"])

temporary = configuration.new_child(dict(font_size=14))
print(temporary["font_size"], configuration["font_size"])

`ChainMap` looks each key up in the dictionaries in the order they were given, hence the priority of the command line over the user, and of the user over the defaults. It is a view, not a copy: the change to `user` shows in it at once, while `merged`, built by `|` (where the last operand wins), kept the values of the moment of the merge. `new_child()` adds a level in front of the others without touching `configuration`. Iteration (`items()`, `dict(configuration)`), however, goes through the dictionaries from the last to the first: the keys of the defaults therefore come out first (`font_size`, `left_margin`, `right_margin`), then `top_margin`.

## Choosing the Right Structure

1. With `timeit.timeit`, measure how long 1,000 membership tests (`-1 in values`) take when `values` is a list of 100,000 integers, then when it is a set.
2. In the same way, measure how long 10,000 removals from the front of 100,000 items take: `pop(0)` on a list, `popleft()` on a `collections.deque`.

Explain the differences you observe with the complexity of each operation.

**Useful documentation**

- [`timeit.timeit`](https://docs.python.org/3/library/timeit.html#timeit.timeit)

In [ ]:
# Your code here

### Additional hints

`timeit.timeit(stmt, setup=..., number=...)` runs `stmt` `number` times and returns the total duration in seconds, not counting `setup`, run once before. `stmt` is a function with no argument (a `lambda`) or a string of code. For the removals from the front, each run empties the list a little: build it in `setup`, so that its construction isn't timed and the measure starts from a full list, and do the 10,000 removals in a single run (`number=1`). `setup` runs in a namespace of its own: it imports what `stmt` uses itself (`"import collections; ..."`).

```python
import collections
import timeit

values_list = list(range(100_000))
values_set = ...
print("list :", timeit.timeit(lambda: -1 in values_list, number=1_000))
print("set :", ...)

print(
  "list.pop(0) :",
  timeit.timeit(
    "for _ in range(10_000): values.pop(0)",
    setup="values = list(range(100_000))",
    number=1,
  ),
)
print("deque.popleft() :", ...)
```

### Solution

In [ ]:
import collections
import timeit

values_list = list(range(100_000))
values_set = set(values_list)
print("list :", timeit.timeit(lambda: -1 in values_list, number=1_000))
print("set :", timeit.timeit(lambda: -1 in values_set, number=1_000))

print(
  "list.pop(0) :",
  timeit.timeit(
    "for _ in range(10_000): values.pop(0)",
    setup="values = list(range(100_000))",
    number=1,
  ),
)
print(
  "deque.popleft() :",
  timeit.timeit(
    "for _ in range(10_000): values.popleft()",
    setup="import collections; values = collections.deque(range(100_000))",
    number=1,
  ),
)

A membership test scans the whole list ($\mathcal{O}(n)$), while a set finds the value through its hash ($\mathcal{O}(1)$): several orders of magnitude apart. Likewise, `pop(0)` shifts every remaining item of the list ($\mathcal{O}(n)$), while `popleft()` removes the first item of a `deque` in $\mathcal{O}(1)$.

## Last Lines of a File

Write a function `tail(path, n)` that returns the last `n` lines of a file, like the Unix `tail` command, without ever keeping more than `n` lines in memory: the file may be much larger than the available memory. Use a `collections.deque`.

Test it on the file created by the next cell: `tail("numbers.txt", 3)` must return `['99997\n', '99998\n', '99999\n']`.

In [ ]:
with open("numbers.txt", "w", encoding="utf8") as fh:
  fh.writelines(f"{i}\n" for i in range(100_000))

In [ ]:
# Your code here

### Additional hints

A file object opened for reading iterates line by line, each line keeping its `\n`. `collections.deque(iterable, maxlen=n)` consumes the iterable keeping only its last `n` items: pass it the file directly, without `readlines()`, which would load the whole file in memory. `list(...)` turns it into the list to return.

```python
import collections


def tail(path: str, n: int) -> list[str]:
  with open(path, encoding="utf8") as fh:
    return list(collections.deque(..., maxlen=...))


tail("numbers.txt", 3)
```

### Solution

In [ ]:
import collections


def tail(path: str, n: int) -> list[str]:
  with open(path, encoding="utf8") as fh:
    return list(collections.deque(fh, maxlen=n))


tail("numbers.txt", 3)

A `deque` with a maximum length of `n` automatically drops the oldest item on each append: it never holds more than `n` lines.

## Inventory

We want to compute the stock of a shop from a list of transactions, named tuples `Transaction(category, item, quantity)` where `quantity` is positive (purchase) or negative (sale).

1. In a single pass over the transactions, compute the inventory `inventory`, the stock of each item of each category, with a `collections.defaultdict` of `collections.Counter`. Converted to plain dictionaries, it must give `{'food': {'beans': 4, 'corn': 5}, 'clothes': {'tshirt': 17}}`.
2. With a method of `Counter`, print the total stock of each category. Expected: `food 9`, then `clothes 17`.

**Useful documentation**

- [`Counter.total`](https://docs.python.org/3/library/collections.html#collections.Counter.total)

In [ ]:
from typing import NamedTuple


class Transaction(NamedTuple):
  category: str
  item: str
  quantity: int


transactions = [
  Transaction("food", "beans", 5),
  Transaction("food", "corn", 10),
  Transaction("clothes", "tshirt", 20),
  Transaction("clothes", "tshirt", -1),
  Transaction("food", "corn", -2),
  Transaction("food", "beans", -1),
  Transaction("clothes", "tshirt", -1),
  Transaction("food", "corn", -3),
  Transaction("clothes", "tshirt", -1),
]


# Your code here

### Additional hints

`defaultdict(Counter)` creates an empty `Counter` the first time a missing category is read, and a `Counter` gives `0` for a missing item: `inventory[category][item] += quantity` therefore works from the first transaction on, with no `if`. A named tuple unpacks like a tuple (`for category, item, quantity in transactions`). `{key: dict(value) for key, value in d.items()}` converts the inventory to plain dictionaries, and `counter.total()` returns the sum of the counts of a `Counter`.

```python
from collections import Counter, defaultdict


inventory = defaultdict(...)
for category, item, quantity in transactions:
  ...


print({category: dict(stock) for category, stock in inventory.items()})
for category, stock in inventory.items():
  print(category, ...)
```

### Solution

In [ ]:
from collections import Counter, defaultdict


inventory = defaultdict(Counter)
for category, item, quantity in transactions:
  inventory[category][item] += quantity


print({category: dict(stock) for category, stock in inventory.items()})
for category, stock in inventory.items():
  print(category, stock.total())

## LRU Cache (Optional)

An LRU (least recently used) cache holds at most `size` values: when full, it makes room by removing the value used the longest time ago. Write a class `LRUCache(size)` that stores its values in a `collections.OrderedDict`, `data`, whose keys go from the least recently used to the most recent:

1. `get(key)` returns the value of `key`, which becomes the most recent, or `None` if `key` is missing;
2. `put(key, value)` adds or replaces the value of `key`, which becomes the most recent; if the cache then exceeds `size` keys, it removes the least recent one.

Test it on a cache of size 2: put `"a"` (1), then `"b"` (2), get `"a"`, then put `"c"` (3). `"b"`, the least recently used key, must be out: `list(cache.data)` must be `['a', 'c']`, and `cache.get("b"), cache.get("a"), cache.get("c")` give `None 1 3`.

In [ ]:
# Your code here

### Additional hints

`data.move_to_end(key)` moves an existing key to the end of the `OrderedDict`, the place of the most recent one, and raises a `KeyError` if it is missing: test `key in data` first, in `get`. An added key already goes to the end, but a replaced key keeps its place: in `put`, move it too. `data.popitem(last=False)` removes the first pair, the least recent one.

```python
import collections


class LRUCache:
  def __init__(self, size: int) -> None:
    self.size = size
    self.data = collections.OrderedDict()

  def get(self, key: str) -> int | None:
    if key not in self.data:
      return None
    ...
    return self.data[key]

  def put(self, key: str, value: int) -> None:
    self.data[key] = value
    ...
    if len(self.data) > self.size:
      ...


cache = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
cache.get("a")
cache.put("c", 3)
print(list(cache.data))
print(cache.get("b"), cache.get("a"), cache.get("c"))
```

### Solution

In [ ]:
import collections


class LRUCache:
  def __init__(self, size: int) -> None:
    self.size = size
    self.data = collections.OrderedDict()

  def get(self, key: str) -> int | None:
    if key not in self.data:
      return None
    self.data.move_to_end(key)
    return self.data[key]

  def put(self, key: str, value: int) -> None:
    self.data[key] = value
    self.data.move_to_end(key)
    if len(self.data) > self.size:
      self.data.popitem(last=False)


cache = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
cache.get("a")
cache.put("c", 3)
print(list(cache.data))
print(cache.get("b"), cache.get("a"), cache.get("c"))

The order of the `OrderedDict` is the order of use: getting `"a"` moved it to the end, so when `"c"` is put, the first key, the one `popitem(last=False)` removes, is `"b"`. Each operation is $\mathcal{O}(1)$, with no scan of the keys. To cache the results of a function, `functools.lru_cache` does the same in a decorator.